# Caruana 그리디 블렌드 — CV 0.5337 제출본 재현

`Models/pairrule_candidates/submission_greedy_g5_dl_pairrule_m3.csv` 를 만드는 노트북이다.
**교차적합 macro F1 0.5336875292416042**, 우리가 가진 후보 중 CV 가 가장 높다.
LB 는 아직 안 받아 봤다.

## 09·10·11 과 무엇이 다른가

앞의 노트북들은 모델을 학습해서 앙상블한다. 이건 **이미 쌓인 OOF 라이브러리에서 고른다.**

```
09  f16 · seed 42 · 사람이 정한 3멤버 0.45/0.45/0.10        CV 0.5165 · LB 0.4818
10  f16 · seed 42/7/2024 · cbopt10 · 9멤버                   CV 0.5210 · LB 0.4725
11  원본 csv 부터 전부 새로 (새 파켓)                        CV 0.5132 · 미제출
12  OOF 라이브러리 98개에서 Caruana 그리디로 49멤버 선택     CV 0.5337 · 미제출  ← 이 노트북
```

재학습이 없다. 행렬 연산이라 몇 분이면 끝난다.

## 그리디가 하는 일

라운드마다 "지금 담긴 것들의 평균에 이 멤버를 하나 더 넣으면 macro F1 이 가장 오르는가"를
보고 하나를 담는다. **복원을 허용**해서 같은 멤버를 여러 번 담을 수 있고, 담긴 횟수가 곧
가중치다. 그래서 멤버 선택과 가중 최적화가 한 번에 끝난다.

정직한 점수는 교차적합으로 낸다 — fold 를 뺀 나머지에서 고르고 그 fold 에서만 잰다.
전체로 한 번에 고르면 0.5382 로 부풀고 그 값으로는 구성을 못 고른다.

## 라이브러리를 고정해야 하는 이유

**그리디 결과는 후보가 무엇이었느냐에 통째로 달려 있다.** 그런데 `artifacts/oof/` 는 실험할
때마다 늘어난다. 실제로 08-05 에 DL OOF 2개를 더 넣었더니 같은 명령이 0.5337 이 아니라
0.5323 을 냈다.

그래서 그때의 목록 98개를 `configs/greedy_g5_dl_library.json` 에 박아 두고 그것만 쓴다.

## 전제 — 기준선 산출물이 필요하다

이 노트북은 **`artifacts/oof/` 의 기존 라이브러리를 읽는다.** `11_full_pipeline.ipynb` 가
만드는 새 실행 폴더가 아니다. 두 쪽 파켓이 달라(`encode_mutation` 정정) 섞으면 안 되므로
아래 셀이 `reset_run_dirs()` 로 기본 경로를 강제한다.

## 0. 준비

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

# 이 노트북은 **기준선** artifacts/oof 의 라이브러리를 쓴다. 11번 노트북이 환경변수를
# 바꿔 놓은 상태로 이어서 돌리면 엉뚱한 폴더를 보므로 여기서 되돌린다.
from cancer_hack.paths import artifacts_dir, process_dir, raw_dir, reset_run_dirs

reset_run_dirs()
RAW, PROC, ART = raw_dir(), process_dir(), artifacts_dir()

LIBRARY_FILE = ROOT / "configs" / "greedy_g5_dl_library.json"
TAG = "nb12_greedy_g5_dl"
MIN_MUT = 3
REFERENCE = ROOT.parent / "Models/pairrule_candidates/submission_greedy_g5_dl_pairrule_m3.csv"

spec = json.loads(LIBRARY_FILE.read_text(encoding="utf-8"))
EXPECTED = spec["expected_crossfit_macro_f1"]

print(f"원본   : {RAW}")
print(f"피처   : {PROC}")
print(f"산출물 : {ART}")
print(f"\n고정 라이브러리 {len(spec['library'])}개 · 기대 교차적합 {EXPECTED:.10f}")
print(f"  n_rounds {spec['n_rounds']} · bag {spec['bag_fraction']}x{spec['bag_rounds']} "
      f"· random_state {spec['random_state']} · {spec['fold_column']}")

원본   : D:\Code\Final_Hachathon\code\data\raw
피처   : D:\Code\Final_Hachathon\code\data\process
산출물 : D:\Code\Final_Hachathon\code\artifacts

고정 라이브러리 98개 · 기대 교차적합 0.5336875292
  n_rounds 30 · bag 0.6x5 · random_state 0 · fold_group5


### fold 와 라이브러리가 그대로인지

OOF 100여 개는 전부 같은 fold 분할에 묶여 있다. fold 를 다시 만들었으면 이 블렌드는
성립하지 않는다. 그리고 고정 목록의 멤버가 하나라도 없으면 그리디가 다른 답을 낸다.

In [2]:
from cancer_hack.provenance import EXPECTED_FOLD_FINGERPRINT, check_fold_fingerprint

print(f"fold 지문 {check_fold_fingerprint(PROC / 'train_folds.parquet')}  "
      f"(기대 {EXPECTED_FOLD_FINGERPRINT}) — 일치")

missing = [n for n in spec["library"] if not (ART / "oof" / f"oof_{n}.csv").exists()]
extra = sorted({p.stem[4:] for p in (ART / "oof").glob("oof_*group5*.csv")} - set(spec["library"]))

print(f"\n고정 목록 {len(spec['library'])}개 중 없는 것 {len(missing)}개")
if missing:
    for name in missing[:5]:
        print(f"  없음: {name[:80]}")
    raise SystemExit("라이브러리가 불완전하다 — 이 블렌드를 재현할 수 없다")
print(f"라이브러리 밖 group5 OOF {len(extra)}개 — 그리디에서 제외된다")
for name in extra[:6]:
    print(f"  제외: {name[:80]}")

fold 지문 997d89a20595cc23  (기대 997d89a20595cc23) — 일치

고정 목록 98개 중 없는 것 0개
라이브러리 밖 group5 OOF 30개 — 그리디에서 제외된다
  제외: catboost_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_
  제외: dl_hybrid_hybrid_svd64_g5_s7_group5
  제외: dl_mlp_mlp_full_svd_g5_s42_group5
  제외: ens11_xcr_group5
  제외: ens11_xcr_group5_raw_blend
  제외: ens11_xcr_group5_uniform_blend


## 1. 그리디 블렌드

`scripts/greedy_blend.py` 를 그대로 부른다 — CLI 와 같은 코드 경로다. `--members-file` 로
라이브러리를 고정하므로 `artifacts/oof/` 가 나중에 늘어나도 결과가 안 흔들린다.

In [3]:
argv = [
    sys.executable, str(ROOT / "scripts/greedy_blend.py"),
    "--cv", spec["cv"],
    "--fold-column", spec["fold_column"],
    "--members-file", str(LIBRARY_FILE),
    "--n-rounds", str(spec["n_rounds"]),
    "--bag-fraction", str(spec["bag_fraction"]),
    "--bag-rounds", str(spec["bag_rounds"]),
    "--random-state", str(spec["random_state"]),
    "--tag", TAG,
]
done = subprocess.run(argv, capture_output=True, text=True, encoding="utf-8", cwd=ROOT)
if done.returncode != 0:
    print(done.stdout[-3000:]); print(done.stderr[-3000:])
    raise SystemExit("greedy_blend 실패")

for line in done.stdout.splitlines():
    if any(k in line for k in ("라이브러리", "검증 통과", "교차적합", "full", "선택된", "제출 후보")):
        print(line)

라이브러리 후보 104개 (group5, 파생 블렌드 제외)
ID·스키마 검증 통과 98개
교차적합 macro F1 = 0.5337   ← 보고할 값
선택된 멤버 49개 / 라이브러리 98개 — 상위 15
  0.065  ( 10회)  단독 0.3957  dl_mlp_mlp_full_features_group5
  0.058  (  9회)  단독 0.4092  dl_set_encoder_gene_set_encoder_full_group5
  0.039  (  6회)  단독 0.3739  dl_hybrid_hybrid_set_mlp_v2_full_features_group5
제출 후보: D:\Code\Final_Hachathon\code\artifacts\submissions\submission_nb12_greedy_g5_dl.csv


### 기대값과 대조

In [4]:
report = json.loads((ART / "logs" / f"{TAG}.json").read_text(encoding="utf-8"))
score = report["crossfit_macro_f1"]

print(f"교차적합 macro F1")
print(f"  기대 {EXPECTED!r}")
print(f"  실측 {score!r}")
print(f"  일치 {score == EXPECTED}")
print(f"\nfold별 {[round(x, 6) for x in report['crossfit_fold_macro_f1']]}")
print(f"라이브러리 {report['library_size']} · 선택 멤버 {len(report['selected'])}개 · "
      f"총 픽 {sum(report['selected'].values())}")

assert score == EXPECTED, "교차적합 점수가 기대와 다르다 — 라이브러리나 fold 를 확인한다"

print("\n가중치 상위 8")
total = sum(report["selected"].values())
for name, count in sorted(report["selected"].items(), key=lambda x: -x[1])[:8]:
    print(f"  {count / total:.3f}  ({count:2d}회)  {name[:74]}")

교차적합 macro F1
  기대 0.5336875292416042
  실측 0.5336875292416042
  일치 True

fold별 [0.531219, 0.534795, 0.529109, 0.509259, 0.540426]
라이브러리 98 · 선택 멤버 49개 · 총 픽 155

가중치 상위 8
  0.090  (14회)  catboost_cbopt10_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl
  0.071  (11회)  rf_repo16_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c
  0.065  (10회)  dl_mlp_mlp_full_features_group5
  0.058  ( 9회)  dl_set_encoder_gene_set_encoder_full_group5
  0.052  ( 8회)  catboost_cbopt10_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl
  0.045  ( 7회)  rf_repo16n_f16n_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c2
  0.045  ( 7회)  rf_repo16n_f16n_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c2
  0.039  ( 6회)  dl_hybrid_hybrid_set_mlp_v2_full_features_group5


## 2. 짝 라벨 규칙

test 행의 유전자 프로파일이 train 의 **유일한** 행과 바이트 단위로 같고 그 train 라벨이
`KIPAN`·`KIRC`·`GBMLGG`·`LGG` 중 하나면 **짝 코호트 라벨**로 바꾼다. 214행이 걸린다.

두 베이스에서 +0.0922·+0.0913 을 벌었다 — 베이스와 거의 무관한 상수 가산이다.
근거는 `docs/pair_rule.md`. 전제가 깨지면 아래 셀이 멈춘다.

In [5]:
from cancer_hack.pair_rule import PAIR, build_pair_rule

rule = build_pair_rule(RAW / "train.csv", RAW / "test.csv", min_mut=MIN_MUT)
violations = rule.verify_premises()
assert violations == [], violations

base_path = ART / "submissions" / f"submission_{TAG}.csv"
base = pd.read_csv(base_path)
final = base.copy()
before = base["SUBCLASS"].to_numpy().copy()
final["SUBCLASS"] = rule.relabel(final["ID"], before)

changed = int((final["SUBCLASS"].to_numpy() != before).sum())
copied = sum(1 for i, c in zip(final["ID"], before)
             if i in rule.mapping and PAIR.get(c) == rule.mapping[i])
print(f"규칙 대상 {rule.diagnostics['n_flipped']}행 · 바뀐 행 {changed} · "
      f"바꾸기 전 train 라벨 복사였던 행 {copied}")

diff = pd.DataFrame({"before": before, "after": final["SUBCLASS"]})
print()
print(diff[diff.before != diff.after].groupby(["before", "after"]).size().to_string())

규칙 대상 214행 · 바뀐 행 214 · 바꾸기 전 train 라벨 복사였던 행 214

before  after 
GBMLGG  LGG       48
KIPAN   KIRC      59
KIRC    KIPAN     57
LGG     GBMLGG    50


## 3. 제출 파일

In [6]:
sample = pd.read_csv(RAW / "sample_submission.csv")
final_path = ART / "submissions" / f"submission_{TAG}_pairrule_m{MIN_MUT}.csv"
final.to_csv(final_path, index=False, encoding="UTF-8-sig")

assert list(final.columns) == ["ID", "SUBCLASS"]
assert len(final) == 2546
assert (final["ID"].astype(str).to_numpy() == sample["ID"].astype(str).to_numpy()).all()
assert final["SUBCLASS"].notna().all()

print(f"규칙 전 : {base_path}")
print(f"최종    : {final_path}")
print(f"  2,546행 · {final['SUBCLASS'].nunique()}클래스 · 스키마 검증 통과")
print(f"\n  교차적합 macro F1 = {score:.4f}   (LB 미검증)")
print("  ※ DACON 업로드는 사람이 직접 한다.")

규칙 전 : D:\Code\Final_Hachathon\code\artifacts\submissions\submission_nb12_greedy_g5_dl.csv
최종    : D:\Code\Final_Hachathon\code\artifacts\submissions\submission_nb12_greedy_g5_dl_pairrule_m3.csv
  2,546행 · 26클래스 · 스키마 검증 통과

  교차적합 macro F1 = 0.5337   (LB 미검증)
  ※ DACON 업로드는 사람이 직접 한다.


## 4. 기준 파일과 대조

08-04 에 만들어 둔 후보와 전 행을 맞춰 본다. 한 행이라도 다르면 이 노트북은 그 파일을
재현하지 못하는 것이다.

In [7]:
if REFERENCE.exists():
    reference = pd.read_csv(REFERENCE)
    same_ids = (reference["ID"].astype(str).to_numpy()
                == final["ID"].astype(str).to_numpy()).all()
    n_diff = int((reference["SUBCLASS"].to_numpy() != final["SUBCLASS"].to_numpy()).sum())
    print(f"기준: {REFERENCE.name}")
    print(f"  ID 순서 일치 : {same_ids}")
    print(f"  다른 행      : {n_diff} / {len(reference)}")
    assert same_ids and n_diff == 0, "제출본을 재현하지 못했다"
    print("\n  재현 확인 — 2,546행 전부 일치")
else:
    print(f"기준 파일이 없어 대조를 건너뛴다: {REFERENCE}")

기준: submission_greedy_g5_dl_pairrule_m3.csv
  ID 순서 일치 : True
  다른 행      : 0 / 2546

  재현 확인 — 2,546행 전부 일치


## 5. 이 후보를 어떻게 볼 것인가

CV 0.5337 은 우리 후보 중 최고다. v002+짝규칙(CV 0.5165 · LB 0.4818)보다 **+0.0172** 높다.

그런데 이 대회에서 CV 순위와 LB 순위가 맞은 적이 거의 없다. LB 점 열 개 중 CV 를 올린
시도는 대부분 LB 에서 떨어졌다.

**그래서 이걸 올리면 배우는 게 크다.** 오르면 CV 를 다시 신뢰할 수 있고, 안 오르면 CV
최적화를 멈추고 다른 축으로 옮겨야 한다는 판단이 선다. 어느 쪽이든 한 번의 제출로
방향이 정해진다.

한 가지 주의 — **이 블렌드는 라이브러리 98개가 며칠에 걸쳐 쌓인 결과물**이다. 개별 멤버를
원본 csv 에서 다시 만들려면 그 실험들을 전부 다시 돌려야 한다. 이 노트북이 재현하는 것은
"그 라이브러리가 주어졌을 때의 선택과 결합"이다. 대회 코드로 낼 때는 09 번(원본 csv 에서
끝까지 가고 LB 로 검증된 경로)을 같이 내는 게 안전하다.